# Grafos — Olist

**Fase 4 de CRISP-DM · módulo de grafos**

Pregunta de negocio: ¿qué productos, categorías o vendedores están más conectados?

Este notebook consume los artefactos generados por `src/models/graphs.py` (no reentrena desde Streamlit).
El grafo principal es el **bipartito categoría–vendedor**, porque es el único 100% conexo y permite
detectar comunidades sin descartar nodos. La co-compra producto–producto se usa como análisis complementario.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "artifacts").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts" / "graphs"

metricas = json.loads((ARTIFACTS / "metricas.json").read_text(encoding="utf-8"))
manifest = json.loads((ARTIFACTS / "manifest.json").read_text(encoding="utf-8"))
print("Modelo ganador:", manifest["winner"])
print("Periodo:", manifest["period"])

## 1. Elección del grafo

Comparamos la conectividad de los tres candidatos. Un grafo con muchos nodos aislados no sirve para
detección de comunidades: Louvain no tiene información para agruparlos.

In [ ]:
resumen = pd.DataFrame([
    {"grafo": "categoría–vendedor", **{k: metricas["grafo_principal"][k] for k in ["nodos", "aristas", "componentes", "componente_gigante_%", "grado_promedio"]}},
    {"grafo": "producto–vendedor", **{k: metricas["grafo_producto_vendedor"][k] for k in ["nodos", "aristas", "componentes", "componente_gigante_%", "grado_promedio"]}},
    {"grafo": "co-compra producto", **{k: metricas["grafo_co_compra"][k] for k in ["nodos", "aristas", "componentes", "componente_gigante_%", "grado_promedio"]}},
    {"grafo": "co-compra categoría", **{k: metricas["grafo_categoria_categoria"][k] for k in ["nodos", "aristas", "componentes", "componente_gigante_%", "grado_promedio"]}},
])
resumen

## 2. Comparación de algoritmos de comunidades

Se comparan Louvain, Greedy Modularity y Label Propagation. El ganador se elige por modularidad;
si la diferencia es ≤ 0.001 (empate técnico), gana el más rápido.

In [ ]:
pd.DataFrame(metricas["comparacion_comunidades"]).sort_values("modularidad", ascending=False)

## 3. Comunidades detectadas

Cada comunidad mezcla categorías y vendedores. Las comunidades grandes con muchas categorías
suelen representar surtidos amplios; las pequeñas, vendedores especializados.

In [ ]:
comunidades = pd.read_csv(ARTIFACTS / "comunidades.csv")
comunidades.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(comunidades.comunidad.astype(str), comunidades.nodos)
ax.set_xlabel("Comunidad")
ax.set_ylabel("Nodos")
ax.set_title("Tamaño de las comunidades (grafo categoría–vendedor)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Centralidad

- **Grado:** popularidad (número de conexiones).
- **PageRank:** importancia considerando la importancia de los vecinos.
- **Betweenness aproximada:** nodos puente entre comunidades.

In [ ]:
print("Categorías con más vendedores:")
display(pd.DataFrame(metricas["top_categorias_por_vendedores"]))
print("Vendedores con más categorías:")
display(pd.DataFrame(metricas["top_vendedores_por_categorias"]))

In [ ]:
print("Productos con mayor PageRank (grafo producto–vendedor):")
display(pd.DataFrame(metricas["top_productos_pagerank"]))
print("Nodos con mayor betweenness aproximada:")
display(pd.DataFrame(metricas["betweenness_aproximada"]))

## 5. Co-compra: análisis complementario

Solo 3,236 pedidos tienen dos o más productos distintos, por lo que este grafo es disperso.
Se usa para detectar oportunidades de venta cruzada, no como base del módulo.

In [ ]:
print("Pares de categorías que más se compran juntas:")
display(pd.DataFrame(metricas["top_pares_categorias"]))
print("Pares de productos que más se compran juntos:")
display(pd.DataFrame(metricas["top_pares_co_compra"]))

## 6. Conclusiones y limitaciones

In [ ]:
for conclusion in manifest["business_conclusions"]:
    print("•", conclusion)
print()
for limitation in manifest["limitations"]:
    print("⚠", limitation)